<a href="https://colab.research.google.com/github/Emboesq13/Curso-Inteligencia-Artificial/blob/main/Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [3]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [18]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [27]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)



En una computadora, **RAM** (memoria de acceso aleatorio) y **almacenamiento** (también llamado memoria secundaria) cumplen funciones muy distintas aunque ambas se refieren a “memoria”. La diferencia principal se resume en **volatilidad, velocidad y propósito**.

| Característica | RAM (Memoria principal) | Almacenamiento (Memoria secundaria) |
|----------------|------------------------|------------------------------------|
| **Volatilidad** | Volátil: pierde su contenido cuando la energía se apaga. | No volátil: conserva la información aunque la computadora esté apagada. |
| **Velocidad** | Muy rápida (nanosegundos). | Más lenta (micros/nanosegundos a milisegundos). |
| **Función** | Almacena los datos y programas que la CPU necesita acceder *en tiempo real* (aplicaciones abiertas, procesos en ejecución). | Guarda los datos y programas de manera *permanente* (sistema operativo, archivos, discos duros, SSD, USB). |
| **Capacidad típica** | Gigabytes (2 GB – 64 GB o más). | Decenas o ci

In [20]:
#Respuesta Completa
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-b45c206c-26a4-451e-8e56-c4997401e70f",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro, SSD, etc.)**\n\n| Característica | RAM | Almacenamiento |\n|----------------|-----|----------------|\n| **Tipo de memoria** | Volátil: pierde su contenido al apagar o reiniciar la máquina. | No volátil: conserva los datos aunque el equipo esté apagado. |\n| **Propósito principal** | Proveer un espacio de trabajo rápido para el procesador: guarda los programas y datos que se están usando en ese momento. | Guardar de forma permanente sistemas operativos, aplicaciones, documentos, fotos, videos, etc. |\n| **Velocidad** | Mucho más rápida (nanosegundos). Las lecturas/escrituras pueden ser 10‑100 veces más rápidas que en un SSD y cientos de veces más que en un HDD. | Más lenta: SSDs ~0.1‑0.5 µs de latencia; HDDs ~5‑10 ms. |\n| **Capaci

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [24]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)

Tokens del prompt: 86
Tokens de la respuesta: 840
Tokens totales: 926


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [25]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")


Tiempo de respuesta: 2.29 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [28]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 2.29 s — 946 tokens
Modelo grande: 2.37 s — 1065 tokens

Respuesta del modelo grande:
 **RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro / SSD)**  

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Tipo de memoria** | Volátil: pierde su contenido cuando se corta la energía. | No volátil: conserva los datos aunque el equipo se apague. |
| **Función principal** | Área de trabajo temporal donde el CPU lee y escribe datos que está procesando en ese momento (programas abiertos, archivos en uso, datos intermedios). | Biblioteca permanente donde se guardan sistemas operativos, aplicaciones, documentos, fotos, videos, etc. |
| **Velocidad** | Muy alta (nanosegundos). Acceso casi instantáneo, lo que permite que el procesador trabaje sin cuellos de botella. | Mucho más lenta (micro‑ a milisegundos). Incluso los SSD son cientos o miles de veces más lentos que la RAM. |
| **Capacidad típica** | 4 GB – 64 GB en la mayoría de ord

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [31]:
# Leer API key desde Colab Secrets
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('Modelos_LLM'))
print("Cliente de Groq inicializado correctamente.")



Cliente de Groq inicializado correctamente.


In [36]:
# Definir la lista de preguntas
preguntas = []
preguntas.append("Por que el disco solido es mejor que el disco duro")
preguntas.append("Una computadora es mas rapida si tiene mas RAM")
preguntas.append("Que procesador es el mejor del mercado")
print(preguntas)

['Por que el disco solido es mejor que el disco duro', 'Una computadora es mas rapida si tiene mas RAM', 'Que procesador es el mejor del mercado']


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [38]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": preguntas[0]}]
)

resultado_1 = {}
inicio = time.time()
resultado_1["Respuesta"] = response.choices[0].message.content
duracion = time.time() - inicio
resultado_1["Tiempo"] = duracion
resultado_1["Tokens"] = response.usage.total_tokens
print(resultado_1)


{'Respuesta': '### ¿Por qué un disco sólido (SSD) suele considerarse “mejor” que un disco duro mecánico (HDD)?\n\n| Criterio | SSD | HDD |\n|----------|-----|-----|\n| **Velocidad de acceso** | < 1\u202fms (latencia) < 0,2\u202fGB\u202f/\u202fs (transferencia) | 5‑15\u202fms (latencia) < 0,1\u202fGB\u202f/\u202fs (transferencia) |\n| **Fiabilidad / Durabilidad** | Sin piezas móviles → menos fallas mecánicas | Piezas móviles → mayor probabilidad de fallo por desgaste |\n| **Consumo de energía** | 3‑5\u202fW (inactivo) | 5‑10\u202fW (inactivo) |\n| **Calor y ruido** | Muy silencioso, bajo calor | Ruidoso, mayor calor |\n| **Tamaño/forma** | 2,5\u202f" o M.2 (compacto) | 3,5\u202f" (tamaño estándar) |\n| **Costo por GB** | Más caro, pero ha bajado 70‑80\u202f% en los últimos 5\u202faños | Menor, sigue siendo la opción más barata a gran escala |\n\n---\n\n## 1. **Velocidad y rendimiento**\n\n| Tipo | Detalle |\n|------|---------|\n| **Latencia de lectura/escritura** | Los SSD llegan a 0,05

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [39]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": preguntas[1]}]
)

resultado_2 = {}
inicio = time.time()
resultado_2["Respuesta"] = response.choices[0].message.content
duracion = time.time() - inicio
resultado_2["Tiempo"] = duracion
resultado_2["Tokens"] = response.usage.total_tokens
print(resultado_2)


{'Respuesta': 'No necesariamente.  \nQue una computadora sea “más rápida” no depende únicamente de la cantidad de RAM, aunque ésta sí influye en muchos casos.  \nA continuación te explico cómo se relaciona y cuándo realmente hace la diferencia:\n\n| Situación | ¿Necesitas más RAM? | ¿Qué pasa sin RAM suficiente? | Factores que pueden limitar el rendimiento |\n|-----------|---------------------|--------------------------------|------------------------------------------|\n| **Aplicaciones de escritorio** (editor de textos, hojas de cálculo, navegadores con muchas pestañas) | Sí, si ya están usando la mayor parte de la memoria disponible (por ejemplo 8\u202fGB en una PC de 16\u202fGB). | El sistema comienza a usar la memoria virtual (swap) en disco, lo que ralentiza drásticamente la respuesta. | Velocidad del disco (HDD vs SSD), CPU, ancho de banda de la memoria. |\n| **Juegos y simulaciones** | Sí, pero con un límite práctico: la mayoría de los juegos actuales funcionan bien con 8–16\u20

In [40]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": preguntas[2]}]
)

resultado_3 = {}
inicio = time.time()
resultado_3["Respuesta"] = response.choices[0].message.content
duracion = time.time() - inicio
resultado_3["Tiempo"] = duracion
resultado_3["Tokens"] = response.usage.total_tokens
print(resultado_3)


{'Respuesta': '### ¿Cuál es “el mejor” procesador del mercado?\n\nLa respuesta depende de **para qué lo vas a usar**:  \n- **Gaming**  \n- **Creación de contenidos (video, 3D, edición, renderizado)**  \n- **Uso general/Multitarea**  \n- **Computación de alto rendimiento (servidores, estaciones de trabajo)**  \n\nA continuación tienes un panorama actualizado (octubre‑2024) con los modelos que dominan cada segmento, y un par de opciones “todo en uno” para comparar.\n\n---\n\n## 1. Gaming (alta frecuencia, buen rendimiento por núcleo)\n\n| CPU | Núcleos / Hilos | Frecuencia base/boost | TDP | Comentarios |\n|-----|-----------------|------------------------|-----|-------------|\n| **AMD Ryzen 9\u202f7950X3D** | 16 / 32 | 4.5\u202fGHz / 5.7\u202fGHz | 170\u202fW | El 3D V-Cache de AMD le da +10‑15\u202f% en títulos que usan caché intensamente. Excelente para 1440p/4K. |\n| **Intel Core i9‑13980HX** | 24 / 32 | 5.0\u202fGHz / 5.5\u202fGHz | 45\u202fW (Mobile) | 24‑núcleo móvil con alta frecu

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [44]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)
print(resultados)


[{'Respuesta': '### ¿Por qué un disco sólido (SSD) suele considerarse “mejor” que un disco duro mecánico (HDD)?\n\n| Criterio | SSD | HDD |\n|----------|-----|-----|\n| **Velocidad de acceso** | < 1\u202fms (latencia) < 0,2\u202fGB\u202f/\u202fs (transferencia) | 5‑15\u202fms (latencia) < 0,1\u202fGB\u202f/\u202fs (transferencia) |\n| **Fiabilidad / Durabilidad** | Sin piezas móviles → menos fallas mecánicas | Piezas móviles → mayor probabilidad de fallo por desgaste |\n| **Consumo de energía** | 3‑5\u202fW (inactivo) | 5‑10\u202fW (inactivo) |\n| **Calor y ruido** | Muy silencioso, bajo calor | Ruidoso, mayor calor |\n| **Tamaño/forma** | 2,5\u202f" o M.2 (compacto) | 3,5\u202f" (tamaño estándar) |\n| **Costo por GB** | Más caro, pero ha bajado 70‑80\u202f% en los últimos 5\u202faños | Menor, sigue siendo la opción más barata a gran escala |\n\n---\n\n## 1. **Velocidad y rendimiento**\n\n| Tipo | Detalle |\n|------|---------|\n| **Latencia de lectura/escritura** | Los SSD llegan a 0,0

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [46]:
# Mostrar la tabla final de resultados

for i, res in enumerate(resultados, 1):
    print(f"Pregunta {i}:")
    print(f"  Respuesta:  {res.get('Respuesta')}")
    print(f"  Tiempo:  {res.get('Tiempo')}")
    print(f"  Tokens:    {res.get('Tokens')}\n")
#El modelo ligero resolvió las 3 preguntas satisfactoriamente,
#demostrando que retiene la información clave y es capaz de seguir
#instrucciones simples sin requerir la capacidad de un modelo de mayor tamaño

Pregunta 1:
  Respuesta:  ### ¿Por qué un disco sólido (SSD) suele considerarse “mejor” que un disco duro mecánico (HDD)?

| Criterio | SSD | HDD |
|----------|-----|-----|
| **Velocidad de acceso** | < 1 ms (latencia) < 0,2 GB / s (transferencia) | 5‑15 ms (latencia) < 0,1 GB / s (transferencia) |
| **Fiabilidad / Durabilidad** | Sin piezas móviles → menos fallas mecánicas | Piezas móviles → mayor probabilidad de fallo por desgaste |
| **Consumo de energía** | 3‑5 W (inactivo) | 5‑10 W (inactivo) |
| **Calor y ruido** | Muy silencioso, bajo calor | Ruidoso, mayor calor |
| **Tamaño/forma** | 2,5 " o M.2 (compacto) | 3,5 " (tamaño estándar) |
| **Costo por GB** | Más caro, pero ha bajado 70‑80 % en los últimos 5 años | Menor, sigue siendo la opción más barata a gran escala |

---

## 1. **Velocidad y rendimiento**

| Tipo | Detalle |
|------|---------|
| **Latencia de lectura/escritura** | Los SSD llegan a 0,05–0,2 ms, mientras que los HDDs rondan 5–10 ms. La diferencia es una reducció